# **ETL Process**

## **Extract**

### **Persiapan Environment**

In [ ]:
!pip -q install pandas numpy scikit-learn joblib matplotlib seaborn

**Import Library**

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

print('Library berhasil dimuat.')

Library berhasil dimuat.


**Unggah file CSV ke Google Colab menggunakan widget upload:**

In [ ]:
from google.colab import files

uploaded = files.upload()   # Klik tombol 'Choose Files', pilih Superstore_Dataset.csv
filename = list(uploaded.keys())[0]
print(f'File diunggah: {filename}')

Saving Superstore Dataset.csv to Superstore Dataset.csv
File diunggah: Superstore Dataset.csv


### **Membaca File CSV**

In [ ]:
df_raw = pd.read_csv(
    filename,
    encoding='latin1',      # Penting: menangani karakter non-UTF8
    parse_dates=False        # Tanggal akan diproses manual di tahap Transform
)

# Tampilkan info dasar dataset
print(f'Jumlah baris   : {df_raw.shape[0]}')
print(f'Jumlah kolom   : {df_raw.shape[1]}')
print(f'\nKolom yang tersedia:')
print(df_raw.columns.tolist())
print(f'\nContoh 3 baris pertama:')
df_raw.head(3)


Jumlah baris   : 9994
Jumlah kolom   : 21

Kolom yang tersedia:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Contoh 3 baris pertama:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


### **Pemeriksaan Awal Data**

In [ ]:
print('=== INFO TIPE DATA ===')
print(df_raw.dtypes)

print('\n=== MISSING VALUES ===')
print(df_raw.isnull().sum())

print('\n=== STATISTIK DESKRIPTIF ===')
print(df_raw[['Sales', 'Quantity', 'Discount', 'Profit']].describe())

print('\n=== NILAI UNIK KOLOM KATEGORIS ===')
for col in ['Ship Mode', 'Segment', 'Region', 'Category']:
    print(f'{col}: {df_raw[col].unique()}')

=== INFO TIPE DATA ===
Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

=== MISSING VALUES ===
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

=

## **Transform**

### **Membuat Salinan Kerja**

In [ ]:
df = df_raw.copy()
print('Salinan data berhasil dibuat.')


Salinan data berhasil dibuat.


### **Data Cleaning**

#### **Standarisasi Nama Kolom**

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

print('Nama kolom setelah standarisasi:')
print(df.columns.tolist())

Nama kolom setelah standarisasi:
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


#### **Konversi Tipe Data**

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], format='%m/%d/%Y')
df['ship_date']  = pd.to_datetime(df['ship_date'],  format='%m/%d/%Y')

# Konversi postal_code ke string (kode pos bukan angka matematis)
df['postal_code'] = df['postal_code'].astype(str).str.zfill(5)

# Verifikasi tipe data setelah konversi
print(df[['order_date', 'ship_date', 'postal_code']].dtypes)
print(df[['order_date', 'ship_date']].head(3))

order_date     datetime64[ns]
ship_date      datetime64[ns]
postal_code            object
dtype: object
  order_date  ship_date
0 2016-11-08 2016-11-11
1 2016-11-08 2016-11-11
2 2016-06-12 2016-06-16


#### **Pengecekan Missing Values & Duplikasi**

In [ ]:
missing = df.isnull().sum()
print('Missing values per kolom:')
print(missing[missing > 0] if missing.any() else 'Tidak ada missing value.')

# ── CELL 10: Cek & hapus duplikasi ──
dup_count = df.duplicated().sum()
print(f'Jumlah baris duplikat: {dup_count}')

if dup_count > 0:
    df = df.drop_duplicates()
    print(f'Duplikat dihapus. Baris tersisa: {len(df)}')
else:
    print('Tidak ada duplikat.')

Missing values per kolom:
Tidak ada missing value.
Jumlah baris duplikat: 0
Tidak ada duplikat.


#### **Membersihkan Data Teks**

In [ ]:
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

print('Whitespace pada kolom teks berhasil dibersihkan.')
print(f'Kolom yang dibersihkan: {list(str_cols)}')

Whitespace pada kolom teks berhasil dibersihkan.
Kolom yang dibersihkan: ['order_id', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name']


### **Feature Engineering**

#### **Shipping Days (Lama Pengiriman)**

In [ ]:
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

print('Distribusi shipping_days:')
print(df['shipping_days'].describe())

Distribusi shipping_days:
count    9994.000000
mean        3.958175
std         1.747567
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         7.000000
Name: shipping_days, dtype: float64


#### **Profit Ratio (Rasio Keuntungan)**

In [ ]:
df['profit_ratio'] = np.where(
    df['sales'] != 0,
    df['profit'] / df['sales'],
    0.0
)
# Bulatkan 4 desimal untuk efisiensi penyimpanan
df['profit_ratio'] = df['profit_ratio'].round(4)

print('Contoh profit_ratio:')
print(df[['sales', 'profit', 'profit_ratio']].head(5))

Contoh profit_ratio:
      sales    profit  profit_ratio
0  261.9600   41.9136        0.1600
1  731.9400  219.5820        0.3000
2   14.6200    6.8714        0.4700
3  957.5775 -383.0310       -0.4000
4   22.3680    2.5164        0.1125


#### **Revenue (Pendapatan Bersih Setelah Diskon)**

In [ ]:
df['revenue'] = (df['sales'] * (1 - df['discount'])).round(2)

print('Contoh revenue:')
print(df[['sales', 'discount', 'revenue']].head(5))

Contoh revenue:
      sales  discount  revenue
0  261.9600      0.00   261.96
1  731.9400      0.00   731.94
2   14.6200      0.00    14.62
3  957.5775      0.45   526.67
4   22.3680      0.20    17.89


#### **Kolom Waktu (Year, Month, Quarter)**

In [ ]:
df['order_year']    = df['order_date'].dt.year
df['order_month']   = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter

# Tambahkan label bulan untuk kemudahan visualisasi
df['month_label'] = df['order_date'].dt.strftime('%b %Y')  # Contoh: 'Nov 2016'

print('Komponen waktu berhasil diekstrak:')
print(df[['order_date', 'order_year', 'order_month', 'order_quarter', 'month_label']].head(5))

Komponen waktu berhasil diekstrak:
  order_date  order_year  order_month  order_quarter month_label
0 2016-11-08        2016           11              4    Nov 2016
1 2016-11-08        2016           11              4    Nov 2016
2 2016-06-12        2016            6              2    Jun 2016
3 2015-10-11        2015           10              4    Oct 2015
4 2015-10-11        2015           10              4    Oct 2015


#### **Profit Flag (Apakah Transaksi Menguntungkan?)**

In [ ]:
df['is_profitable'] = (df['profit'] > 0).astype(int)  # 1 = untung, 0 = rugi/impas

profitable = df['is_profitable'].sum()
total = len(df)
print(f'Transaksi menguntungkan: {profitable} / {total} ({profitable/total*100:.1f}%)')

Transaksi menguntungkan: 8058 / 9994 (80.6%)


#### **Validasi Akhir Data**

In [ ]:
print('=== RINGKASAN DATASET SETELAH TRANSFORM ===')
print(f'Jumlah baris  : {len(df)}')
print(f'Jumlah kolom  : {df.shape[1]}')
print(f'Kolom baru    : shipping_days, profit_ratio, revenue,')
print(f'                order_year, order_month, order_quarter,')
print(f'                month_label, is_profitable')
print()
print('Tipe data final:')
print(df.dtypes)

# Simpan dataset bersih ke CSV
df.to_csv('superstore_clean.csv', index=False)
print('\nDataset bersih disimpan ke superstore_clean.csv')

=== RINGKASAN DATASET SETELAH TRANSFORM ===
Jumlah baris  : 9994
Jumlah kolom  : 29
Kolom baru    : shipping_days, profit_ratio, revenue,
                order_year, order_month, order_quarter,
                month_label, is_profitable

Tipe data final:
row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code              object
region                   object
product_id               object
category                 object
sub_category             object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
shipping_days             int64
profit_ratio            float64
revenue  

## **Load**

#### **Perbaikan Duplikat Dimension Key**

In [ ]:
# Buat canonical mapping (ambil data pertama per key)
prod_canonical = df[['product_id','product_name','category','sub_category']]\
    .groupby('product_id', as_index=False).first()

loc_canonical  = df[['postal_code','city','state','region','country']]\
    .groupby('postal_code', as_index=False).first()

# Update df dengan nilai canonical
df = df.merge(prod_canonical[['product_id','product_name']]\
    .rename(columns={'product_name':'pn_fix'}), on='product_id', how='left')
df['product_name'] = df['pn_fix']
df.drop(columns=['pn_fix'], inplace=True)

df = df.merge(loc_canonical[['postal_code','city']]\
    .rename(columns={'city':'city_fix'}), on='postal_code', how='left')
df['city'] = df['city_fix']
df.drop(columns=['city_fix'], inplace=True)

print('Verifikasi (max harus = 1):')
print('product_id:', df.groupby('product_id')['product_name'].nunique().max())
print('postal_code:', df.groupby('postal_code')['city'].nunique().max())

Verifikasi (max harus = 1):
product_id: 1
postal_code: 1


#### **Membuat Tabel Dimensi & Fakta**

In [ ]:
# ── CELL 19: Buat dimension tables ──
dim_customers = df[['customer_id','customer_name','segment']]\
    .drop_duplicates().reset_index(drop=True)
dim_customers.insert(0, 'customer_pk', range(1, len(dim_customers)+1))

dim_products = df[['product_id','product_name','category','sub_category']]\
    .drop_duplicates().reset_index(drop=True)
dim_products.insert(0, 'product_pk', range(1, len(dim_products)+1))

dim_location = df[['postal_code','city','state','region','country']]\
    .drop_duplicates().reset_index(drop=True)
dim_location.insert(0, 'location_pk', range(1, len(dim_location)+1))

dates = df[['order_date']].drop_duplicates().copy()
dates['date_pk']     = dates['order_date'].dt.strftime('%Y%m%d').astype(int)
dates['year']        = dates['order_date'].dt.year
dates['month']       = dates['order_date'].dt.month
dates['quarter']     = dates['order_date'].dt.quarter
dates['month_name']  = dates['order_date'].dt.strftime('%B')
dates['day_of_week'] = dates['order_date'].dt.day_name()
dim_date = dates.rename(columns={'order_date':'full_date'}).reset_index(drop=True)


In [ ]:
# ── CELL 20: Buat fact table dengan FK ──
df_fact = df.merge(dim_customers[['customer_id','customer_pk']], on='customer_id', how='left')
df_fact = df_fact.merge(dim_products[['product_id','product_pk']],   on='product_id',  how='left')
df_fact = df_fact.merge(dim_location[['postal_code','location_pk']], on='postal_code',  how='left')
df_fact['date_pk'] = df_fact['order_date'].dt.strftime('%Y%m%d').astype(int)

fact_orders = df_fact[[
    'row_id','order_id','date_pk','ship_date','ship_mode',
    'customer_pk','product_pk','location_pk',
    'sales','quantity','discount','profit',
    'shipping_days','profit_ratio','revenue','is_profitable'
]].copy()

print('fact_orders shape:', fact_orders.shape)


fact_orders shape: (9994, 16)


#### **Membuat File SQL**

In [ ]:
def escape_sql(value):
    if isinstance(value, str):
        value = value.replace('\\', '\\\\')
        value = value.replace("'", "\\'")
    return value

def df_to_insert_sql(df, table_name, batch_size=500):
    sql_lines = []
    cols = ', '.join([f'`{c}`' for c in df.columns])
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start+batch_size]
        values_list = []
        for _, row in batch.iterrows():
            vals = []
            for v in row:
                if pd.isna(v) if not isinstance(v, str) else False: vals.append('NULL')
                elif isinstance(v, str): vals.append(f"'{escape_sql(v)}'")
                elif hasattr(v, 'strftime'): vals.append(f"'{v.strftime('%Y-%m-%d')}'")
                elif isinstance(v, (int, np.integer)): vals.append(str(int(v)))
                else: vals.append(str(round(float(v), 4)))
            values_list.append(f'({", ".join(vals)})')
        sql_lines.append(f'INSERT INTO `{table_name}` ({cols}) VALUES\n' +
                          ',\n'.join(values_list) + ';')
    return '\n\n'.join(sql_lines)

In [ ]:
DDL = """
SET NAMES utf8mb4;
SET FOREIGN_KEY_CHECKS = 0;
SET SQL_MODE = "NO_AUTO_VALUE_ON_ZERO";

DROP DATABASE IF EXISTS superstore_dw;
CREATE DATABASE superstore_dw
  CHARACTER SET utf8mb4
  COLLATE utf8mb4_unicode_ci;
USE superstore_dw;

CREATE TABLE `dim_customers` (
    `customer_pk`   INT          NOT NULL AUTO_INCREMENT,
    `customer_id`   VARCHAR(20)  NOT NULL,
    `customer_name` VARCHAR(100) NOT NULL,
    `segment`       VARCHAR(30)  NOT NULL,
    PRIMARY KEY (`customer_pk`),
    UNIQUE KEY `uq_customer_id` (`customer_id`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

CREATE TABLE `dim_products` (
    `product_pk`   INT          NOT NULL AUTO_INCREMENT,
    `product_id`   VARCHAR(30)  NOT NULL,
    `product_name` VARCHAR(300) NOT NULL,
    `category`     VARCHAR(50)  NOT NULL,
    `sub_category` VARCHAR(50)  NOT NULL,
    PRIMARY KEY (`product_pk`),
    UNIQUE KEY `uq_product_id` (`product_id`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

CREATE TABLE `dim_location` (
    `location_pk` INT          NOT NULL AUTO_INCREMENT,
    `postal_code` VARCHAR(10)  NOT NULL,
    `city`        VARCHAR(100) NOT NULL,
    `state`       VARCHAR(100) NOT NULL,
    `region`      VARCHAR(30)  NOT NULL,
    `country`     VARCHAR(50)  NOT NULL,
    PRIMARY KEY (`location_pk`),
    UNIQUE KEY `uq_postal_code` (`postal_code`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

CREATE TABLE `dim_date` (
    `date_pk`     INT         NOT NULL,
    `full_date`   DATE        NOT NULL,
    `year`        SMALLINT    NOT NULL,
    `month`       TINYINT     NOT NULL,
    `quarter`     TINYINT     NOT NULL,
    `month_name`  VARCHAR(15) NOT NULL,
    `day_of_week` VARCHAR(15) NOT NULL,
    PRIMARY KEY (`date_pk`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

CREATE TABLE `fact_orders` (
    `row_id`        INT            NOT NULL,
    `order_id`      VARCHAR(20)    NOT NULL,
    `date_pk`       INT            NOT NULL,
    `ship_date`     DATE           NOT NULL,
    `ship_mode`     VARCHAR(30)    NOT NULL,
    `customer_pk`   INT            NOT NULL,
    `product_pk`    INT            NOT NULL,
    `location_pk`   INT            NOT NULL,
    `sales`         DECIMAL(10,2)  NOT NULL,
    `quantity`      SMALLINT       NOT NULL,
    `discount`      DECIMAL(4,2)   NOT NULL,
    `profit`        DECIMAL(10,4)  NOT NULL,
    `shipping_days` TINYINT        NOT NULL,
    `profit_ratio`  DECIMAL(6,4)   NOT NULL,
    `revenue`       DECIMAL(10,2)  NOT NULL,
    `is_profitable` TINYINT(1)     NOT NULL DEFAULT 0,
    PRIMARY KEY (`row_id`),
    KEY `idx_date`     (`date_pk`),
    KEY `idx_customer` (`customer_pk`),
    KEY `idx_product`  (`product_pk`),
    KEY `idx_location` (`location_pk`),
    CONSTRAINT `fk_date`     FOREIGN KEY (`date_pk`)     REFERENCES `dim_date`(`date_pk`),
    CONSTRAINT `fk_customer` FOREIGN KEY (`customer_pk`) REFERENCES `dim_customers`(`customer_pk`),
    CONSTRAINT `fk_product`  FOREIGN KEY (`product_pk`)  REFERENCES `dim_products`(`product_pk`),
    CONSTRAINT `fk_location` FOREIGN KEY (`location_pk`) REFERENCES `dim_location`(`location_pk`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

SET FOREIGN_KEY_CHECKS = 1;
"""

print("DDL siap.")

DDL siap.


In [ ]:
dim_date_sql = dim_date.copy()
dim_date_sql['full_date'] = dim_date_sql['full_date'].dt.strftime('%Y-%m-%d')
fact_sql = fact_orders.copy()
fact_sql['ship_date'] = fact_sql['ship_date'].dt.strftime('%Y-%m-%d')

# (DDL tersedia di file fixed_etl_superstore.py)
full_sql = '\n\n'.join([DDL,
    '-- dim_customers\n' + df_to_insert_sql(dim_customers, 'dim_customers'),
    '-- dim_products\n'  + df_to_insert_sql(dim_products,  'dim_products'),
    '-- dim_location\n'  + df_to_insert_sql(dim_location,  'dim_location'),
    '-- dim_date\n'      + df_to_insert_sql(dim_date_sql,  'dim_date'),
    '-- fact_orders\n'   + df_to_insert_sql(fact_sql,      'fact_orders'),
])

with open('superstore_dw_fixed.sql', 'w', encoding='utf-8') as f:
    f.write(full_sql)
files.download('superstore_dw_fixed.sql')
print('✅ File SQL berhasil dibuat dan didownload!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ File SQL berhasil dibuat dan didownload!
